# Notebook 2 — Phase 2 mid-train: QLoRA / LoRA on top of `oars344/attnres-phase1`**Repo-driven launcher** — Phase 2 lives in `src/training/qlora.py` (bitsandbytes 4-bit+ LoRA helpers) and `src/training/train_phase2.py` (HF Trainer CLI).The notebook only orchestrates; all real work happens in those modules.What the notebook does:1. Install + GPU probe + HF auth check.2. Clone the repo and `pip install -e .`.3. Load `oars344/attnres-phase1` via `AutoModelForCausalLM` (auto-routed   because `src/model/hf_wrapper.py` registers `model_type="attnres"`).4. Verify the **nine AttnRes-aware LoRA targets** are present in the   base: the seven standard Llama linears **plus** `attn_res.proj` and   `mlp_res.proj` (BlockAttnRes pseudo-query projections).5. Apply LoRA with r=16, α=32, dropout=0.05 and `print_trainable_parameters()`   must report non-zero trainable rows for `attn_res.proj` / `mlp_res.proj`   (headline assertion that LoRA actually touches the residual-stream   projections, not just the standard attention linears).6. Run `--smoke` (50 steps, ~5 min) — the cheap full-stack check.7. Run full Phase 2 (`--max-steps 2000 --save-interval 200`) + push LoRA   adapters to HF Hub.8. Plot the loss curve + sanity-check the adapters re-load.**Runtime budgets** (rough):- Single T4: 50-step smoke ≈ 5 min · 2000-step full ≈ 5–7 hrs.- 2×T4 DDP: 50-step smoke ≈ 3 min · 2000-step full ≈ 2–4 hrs.- Modal A10G: order of magnitude faster (~30 min for the full run).Use `--save-interval 200` (the Phase 2 default) so HF push survivesdisconnects — no manual checkpoint plumbing needed.---

## Step 1 — Install runtime deps, GPU probe, HF auth checkStandalone runtime dependencies (the `attnres-lm` package is installedin the next cell, after cloning). Note `bitsandbytes` is included butthe default Phase 2 config keeps `quantize_base: false` (our 114M basefits in bf16 without 4-bit quant); pass `--quantize-base` at the traincell if you scale to a larger base.

In [ ]:
!pip install -q 'transformers>=4.40,<5' 'peft>=0.7,<0.19' accelerate datasets huggingface_hub safetensors pyyaml tiktoken bitsandbytes sentencepiece# STEP A — shell-level GPU probe via nvidia-smi. Fast and runs BEFORE# we import torch. If torch.cuda hangs on a bad CUDA binding, this# output tells you whether the problem is 'no GPU attached' vs 'torch bindings stuck'.!nvidia-smi# STEP B — now safe to import torch.import torch, os, sysprint(f"PyTorch {torch.__version__}  CUDA {torch.version.cuda}")if torch.cuda.is_available():    n_gpus = torch.cuda.device_count()    for i in range(n_gpus):        print(f"GPU {i}: {torch.cuda.get_device_name(i)}  VRAM: {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")else:    print('\n\n=== GPU problem identified by diagnostic ===\n')    print('If !nvidia-smi above showed a real GPU line, but torch.cuda.is_available() is False, then PyTorch CUDA bindings are stuck.')    print('  FIX: Runtime > Disconnect and delete runtime, then reconnect with T4 GPU selected. Re-run this cell.')    print('If !nvidia-smi above was empty or errored, no GPU is attached at the driver level.')    print('  FIX: Runtime > Change runtime type > T4 GPU, then reconnect. Re-run this cell.')    print('Either way, do NOT just retry this cell - the broken state is sticky. Force-restart the runtime.\n')    sys.exit('aborting: GPU not available, fix above and re-run')from huggingface_hub import HfApiif os.getenv('HF_TOKEN'):    api = HfApi(token=os.environ['HF_TOKEN'])    me = api.whoami()    print(f"HF auth OK - signed in as {me['name']}")else:    print('HF_TOKEN not found - adapters will stay local at the end.')    print('Set it via: %env HF_TOKEN=hf_xxx   (in Colab) or export HF_TOKEN=... (locally)')

## Step 2 — Clone the repo and install the `attnres-lm` packageAfter this cell you can `from src.model.hf_wrapper import AttnResLMForCausalLM`— Phase 2 routes to the right class because `hf_wrapper.py` auto-registers`model_type="attnres"` with `transformers.AutoConfig` at module-import time.

In [ ]:
REPO_URL = "https://github.com/Mateooo93/llm-coding-train.git"LOCAL_DIR = "/content/attnres-lm"import os, sys# If a previous attempt already cloned an EMPTY repo to LOCAL_DIR, fetch with# --unshallow and hard-reset to origin/HEAD (with origin/main fallback) so a# re-run after you push brings in the freshly-pushed code. Preserves LOCAL_DIR# itself (in-session tweaks are visible in the diff before reset).if os.path.isdir(LOCAL_DIR):    !cd {LOCAL_DIR} && git fetch --unshallow origin || git fetch origin; git remote set-head origin --auto; git reset --hard origin/HEAD || git reset --hard origin/main    print(f'Reset {LOCAL_DIR} to origin/HEAD (any local edits dropped).')else:    !git clone --depth 1 {REPO_URL} {LOCAL_DIR}%cd {LOCAL_DIR}# === GUARD: detect empty-clone scenario BEFORE pip install ===if not os.path.exists('setup.py') or not os.path.isdir('src'):    print('\n\n=== DIAGNOSTIC: empty clone — code has not been pushed yet ===\n')    print(f'Cloned {REPO_URL} but the repo is empty (only LICENSE/README/.gitignore).')    print('You need to push the code from a GitHub-authenticated local shell:\n')    print('  ## On your laptop / local machine (with github.com password, PAT, or SSH set up):\n')    print('  cd <your-local-clone-path>      # the path where you cloned this repo on your laptop (e.g. ~/code/llm-coding)')    print('  git add -A && git commit -m "Phase 2 training code"')    print('  git push -u origin main\n')    print('After pushing, re-run THIS cell from the top.')    sys.exit('Aborting: push your code first, then clone+install again.')!pip install -e . --quiet# Smoke-test the package imports (especially the HF wrapper registration).from src.model.model import AttnResLMfrom src.model.hf_wrapper import AttnResLMConfigHF, AttnResLMForCausalLMfrom src.training.train_phase2 import main as phase2_mainprint(f"attnres-lm package ready: AttnResLM={AttnResLM.__module__}.{AttnResLM.__name__}")print(f"HF wrapper: AttnResLMForCausalLM={AttnResLMForCausalLM.__module__}.{AttnResLMForCausalLM.__name__}")print(f"Phase 2 entry point: src.training.train_phase2.main")

## Step 3 — Load the AttnResLM base from HF Hub + verify the LoRA targetsLoading the base from a public HF repo via `AutoModelForCausalLM.from_pretrained`ONLY works because `src/model/hf_wrapper.py` registers`AutoConfig.register("attnres", AttnResLMConfigHF)` at module-import time.Without that, you would need `trust_remote_code=True` or our wrapperimported manually.After this cell, `model.model.layers[i].attn_res.proj.weight` is a`[1, hidden_size]` tensor — the learned pseudo-query projection for thei-th block's residual aggregation. PEFT matches LoRA target_modulesexactly against these leaf names, so we **must** verify they're presentbefore claiming AttnRes-aware LoRA actually attaches to them.

In [ ]:
# === transformers.utils compat shim (re-publish is_*_available for ===# peft 0.18- / transformers 5.x that still expect them at transformers.utils)import transformers.utils as _tutry:    from transformers.utils import import_utils as _tuiexcept ImportError:    _tui = None_REEXPORT = ('is_tf_available','is_torch_available','is_flax_available','is_tokenizers_available','is_remote_url')for _name in _REEXPORT:    if not hasattr(_tu, _name):        if _tui is not None and hasattr(_tui, _name):            setattr(_tu, _name, getattr(_tui, _name))        else:            setattr(_tu, _name, (lambda _f=False: _f))print('transformers.utils compat shim: applied')import torchfrom transformers import AutoModelForCausalLM, AutoTokenizerBASE_ID = "oars344/attnres-phase1"print(f"Loading base from HF Hub: {BASE_ID}")tokenizer = AutoTokenizer.from_pretrained(BASE_ID)print(f"Tokenizer: vocab={tokenizer.vocab_size}, model_max_length={tokenizer.model_max_length}")model = AutoModelForCausalLM.from_pretrained(    BASE_ID,    torch_dtype=torch.bfloat16,    device_map="auto",)print(f"\nBase loaded: {type(model).__module__}.{type(model).__name__}")n_params = sum(p.numel() for p in model.parameters()) / 1e6print(f"  total parameters: {n_params:.1f}M")# AttnRes-aware LoRA target list — seven standard Llama linears plus the# two BlockAttnRes pseudo-query projections. PEFT suffix-matches against# the leaf names of nn.Linear modules inside the wrapped model.attnres_targets = [    "q_proj", "k_proj", "v_proj", "o_proj",    "gate_proj", "up_proj", "down_proj",    "attn_res.proj", "mlp_res.proj",]matched = {t: 0 for t in attnres_targets}for name, mod in model.named_modules():    if isinstance(mod, torch.nn.Linear):        for t in attnres_targets:            if name.endswith(t):                matched[t] += 1print("\nMatched AttnRes-aware LoRA target counts (across all layers):")for t, c in matched.items():    print(f"  {t:>20}: {c}")missing = [t for t, c in matched.items() if c == 0]assert not missing, f"Missing target_modules in the model: {missing}"print("\n[OK] All 9 AttnRes-aware LoRA targets present")# Probe the actual BlockAttnRes pseudo-query projections.print("\nPseudo-query projection shapes:")for i in range(min(3, len(model.model.layers))):    proj_a = model.model.layers[i].attn_res.proj.weight    proj_m = model.model.layers[i].mlp_res.proj.weight    print(f"  layer {i}: attn_res.proj={tuple(proj_a.shape)}  mlp_res.proj={tuple(proj_m.shape)}")

## Step 4 — Apply LoRA on the AttnRes-aware targetsHeadline assertion of this notebook: PEFT's `print_trainable_parameters()`must report a **non-zero** number of trainable params for `q_proj`,`k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj` ANDfor `attn_res.proj` and `mlp_res.proj`. If `attn_res.proj`/`mlp_res.proj`show 0 trainable rows, the LoRA twist didn't attach to theBlockAttnRes pseudo-query projections and you won't actually be re-routingthe residual stream — add the targets explicitly via`LoraConfig(target_modules=[...])` (they're in our default) or check thatyou're loading the right base.Note: default config keeps `quantize_base: false` (the 114M AttnResLMfits comfortably in bf16). To scale to a bigger base later, pass`--quantize-base` to the train cell and `bitsandbytes` 4-bit will engage.

In [ ]:
from peft import LoraConfig, get_peft_modellora = LoraConfig(    r=16,    lora_alpha=32,    lora_dropout=0.05,    bias="none",    task_type="CAUSAL_LM",    target_modules=attnres_targets,)model = get_peft_model(model, lora)model.print_trainable_parameters()

## Step 5 — Run `--smoke` (50 steps, ~5 min) for full-stack validationAlways run the smoke before committing to a multi-hour full run. Thesmoke exercises:- HF Hub base download + AutoModel routing via `AttnResLMConfigHF`- LoRA target_modules matching against `attn_res.proj` / `mlp_res.proj`- HF Trainer init + first 50 forward+backward steps- Streaming dataset (FineWeb-Edu) tokenization round-tripIf `--smoke` fails, the full run will fail too — fix here, not afterhours of wasted GPU time.

In [ ]:
import osn_gpus = torch.cuda.device_count()launcher = f"torchrun --nproc_per_node={n_gpus}" if n_gpus > 1 else "python"# HF username for repo-id (smoke keeps no push; HF_TOKEN presence is also# used by the train script to decide whether to attempt push at all).if os.getenv('HF_TOKEN'):    from huggingface_hub import HfApi    me = HfApi(token=os.environ['HF_TOKEN']).whoami()    hf_user = me['name']else:    hf_user = "local-user"cmd = [    launcher, "-m", "src.training.train_phase2",    "--smoke",    "--base-model", "oars344/attnres-phase1",    "--config", "config/phase2_qlora.yaml",    "--output-dir", "/content/attnres-runs/phase2_smoke",    "--seq-len", "512",    "--batch-size", "1",    "--grad-accum", str(max(1, 16 // max(1, n_gpus))),    "--gc",    "--amp-dtype", "bf16",]import shlexprint(f"\n=== Smoke launch ===\n{' '.join(shlex.quote(c) for c in cmd)}\n")!{' '.join(cmd)}import os.path as _osplosses_path = "/content/attnres-runs/phase2_smoke/losses.json"if _osp.exists(losses_path):    print(f"\nlosses.json ready at {losses_path} - Cell 8 will plot it.")

## Step 6 — Run full Phase 2 (2000 steps) + push LoRA adapters to HF HubDefaults pulled from `config/phase2_qlora.yaml`. Override any of thesehere without editing the YAML:- `--max-steps N` — total optimizer steps (default 2000). Lower for fast  iterations, higher for stronger fine-tunes.- `--batch-size`, `--grad-accum` — effective batch = `batch * grad_accum * n_gpus`.- `--save-interval N` — checkpoints every N steps. Default 200 (10 ckpts  across a 2000-step run; ~330 MB total adapter storage; recover from  disconnects in ≤45 min of work lost).- `--quantize-base` — apply bitsandbytes 4-bit to the base (only useful  for larger bases; the 114M `oars344/attnres-phase1` fits in bf16).- `--lora-r`, `--lora-alpha`, `--lora-dropout` — override LoRA hyperparams.- `--max-grad-norm`, `--learning-rate`, `--warmup-ratio` — trainer knobs.Resume-from-checkpoint works out of the box: HF `Trainer.train`auto-detects the latest checkpoint in `--output-dir` when re-running.

In [ ]:
import osn_gpus = torch.cuda.device_count()launcher = f"torchrun --nproc_per_node={n_gpus}" if n_gpus > 1 else "python"if os.getenv('HF_TOKEN'):    from huggingface_hub import HfApi    me = HfApi(token=os.environ['HF_TOKEN']).whoami()    hf_user = me['name']    phase2_repo = f"{hf_user}/attnres-phase2"else:    phase2_repo = "local-user/attnres-phase2"cmd = [    launcher, "-m", "src.training.train_phase2",    "--base-model", "oars344/attnres-phase1",    "--config", "config/phase2_qlora.yaml",    "--output-dir", "/content/attnres-runs/phase2",    "--max-steps", "2000",    "--save-interval", "200",    "--seq-len", "1024",    "--batch-size", "1",    "--grad-accum", str(max(1, 16 // max(1, n_gpus))),    "--gc",    "--amp-dtype", "bf16",]if os.getenv('HF_TOKEN'):    cmd += ["--push-to-hub", "--repo-id", phase2_repo]import shlexprint(f"\n=== Phase 2 launch ===\n{' '.join(shlex.quote(c) for c in cmd)}\n")!{' '.join(cmd)}print(f"\n[final] adapters: https://huggingface.co/{phase2_repo}")

## Step 7 — Loss curve + adapter reload sanity checkTwo things worth checking after the full run:1. Loss actually went DOWN (sanity: we're not just memorizing).2. The saved adapters re-load cleanly and `print_trainable_parameters()`   again reports non-zero rows for `attn_res.proj` / `mlp_res.proj`.   This is the round-trip test that LoRA saved what we *think* it saved.

In [ ]:
import os, jsonimport matplotlib.pyplot as pltlosses_path = "/content/attnres-runs/phase2/losses.json"if os.path.exists(losses_path):    with open(losses_path) as f:        data = json.load(f)    losses = data.get('step', data.get('steps', []))    if isinstance(losses, list) and len(losses) >= 2:        plt.figure(figsize=(8, 4))        plt.plot(losses[::20], alpha=0.3, label='raw (every 20)')        k = 50        smoothed = [            sum(losses[max(0, i-k+1):i+1]) / (i - max(0, i-k+1) + 1)            for i in range(len(losses))        ]        plt.plot(smoothed, color='C1', label=f'smoothed (k={k})')        plt.xlabel('step'); plt.ylabel('loss')        plt.title('Phase 2 AttnResLM LoRA fine-tune - training loss')        plt.grid(alpha=0.3); plt.legend(); plt.show()        print(f"first 50-step avg: {sum(losses[:50])/min(50, len(losses)):.4f}")        print(f"last  50-step avg: {sum(losses[-50:])/min(50, len(losses)):.4f}")    else:        print(f"losses.json at {losses_path} has no usable 'step' list.")else:    print(f"{losses_path} not present — re-run after Cell 7 produces it.")# Adapter reload — the round-trip test.import torchfrom peft import PeftModelphase2_repo = f"{hf_user}/attnres-phase2" if os.getenv('HF_TOKEN') else "local-user/attnres-phase2"adapter_dir = "/content/attnres-runs/phase2/final"if os.path.isdir(adapter_dir):    print(f"\nReloading adapters from local {adapter_dir} ...")    reloaded = PeftModel.from_pretrained(        AutoModelForCausalLM.from_pretrained(            "oars344/attnres-phase1",            torch_dtype=torch.bfloat16,            device_map="auto",        ),        adapter_dir,    )    reloaded.print_trainable_parameters()

## Done — what you provedIf the smoke cell printed `step_loss` values that go DOWN, and the`trainable params` line named BOTH the seven standard linears AND`attn_res.proj`/`mlp_res.proj`, you've validated:1. **Phase 2 base loads via `AutoModel` without `trust_remote_code=True`**   because `src/model/hf_wrapper.py` auto-registers `model_type="attnres"`   at import time.2. **LoRA actually attaches to the BlockAttnRes pseudo-query projections**   (not just standard attention linears) — this is what makes Phase 2 a   *task-specific residual re-router* and not just another attention-only   LoRA.3. **QLoRA / LoRA plumbing works end-to-end on our AttnResLM** — HF Trainer,   gradient checkpointing, streaming dataset, mixed precision.4. **Adapters persist through the HF Hub round-trip** — they're loadable   back from `oars344/attnres-phase2/...` and the trainable-params line   still names the AttnRes projections.## Cost summarySmoke: ~5 min on T4 (~$0). Full: ~5–7 hrs (~$0 on Colab, ~$5 on Modal A10G).## Next: evaluationRun MMLU + HumanEval on the `oars344/attnres-phase2` adapter to quantifythe gain over the AttnResLM base.When you're ready, push your trained notebook (with outputs visible) toa HF dataset repo so the run is repro-able from anywhere.